# Race analysis notebook

This notebook moves beyond the final results to understand what happens during a race:
how drivers and teams compare in pace, how tyre strategies and pit stops shape the outcome,
and how positions evolve lap after lap. The goal is to explain how the race was won (and lost).

The 2024 Italian Grand Prix is used as the case study: Charles Leclerc won with a one-stop
strategy against the two-stop strategy of the McLarens, which makes it a clear example of
a race decided by tyre management and strategy. The same analysis can be applied to any
race available in FastF1.

## Setup

Unlike the exploration notebook, FastF1 is not used directly here: sessions are loaded
through `load_session` from the project's `f1analysis.data` module, which enables the
cache and loads the race by default. The race is identified by year and `RoundNumber`.
Telemetry is not loaded at this stage and is requested only where needed.

In [ ]:
%load_ext autoreload
%autoreload 2

import fastf1
import pandas as pd
import plotly.express as px
import fastf1.plotting
import numpy as np

from f1analysis.cleaning import clean_laps
from f1analysis.data import load_session

fastf1.set_log_level("WARNING")
pd.set_option("display.max_columns", None)

race = load_session(2024, 16)   # 2024 Italian Grand Prix, race
compound_colors = fastf1.plotting.get_compound_mapping(session=race)
race

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2024 Season Round 16: Italian Grand Prix - Race

## Race result

The final classification is the starting point: it shows the outcome that the rest of
the notebook tries to explain.

In [2]:
race.results[["Abbreviation", "TeamName", "GridPosition", "Position", "Status", "Time", "Points"]]

,Abbreviation,TeamName,GridPosition,Position,Status,Time,Points
16,LEC,Ferrari,4.0,1.0,Finished,0 days 01:14:40.727000,25.0
81,PIA,McLaren,2.0,2.0,Finished,0 days 00:00:02.664000,18.0
4,NOR,McLaren,1.0,3.0,Finished,0 days 00:00:06.153000,16.0
55,SAI,Ferrari,5.0,4.0,Finished,0 days 00:00:15.621000,12.0
44,HAM,Mercedes,6.0,5.0,Finished,0 days 00:00:22.820000,10.0
1,VER,Red Bull Racing,7.0,6.0,Finished,0 days 00:00:37.932000,8.0
63,RUS,Mercedes,3.0,7.0,Finished,0 days 00:00:39.715000,6.0
11,PER,Red Bull Racing,8.0,8.0,Finished,0 days 00:00:54.148000,4.0
23,ALB,Williams,9.0,9.0,Finished,0 days 00:01:07.456000,2.0
20,MAG,Haas F1 Team,13.0,10.0,Finished,0 days 00:01:08.302000,1.0


Leclerc won from 4th on the grid, only 2.7 seconds ahead of Piastri. Norris started from
pole position but finished 3rd, despite scoring the extra point for the fastest lap
(16 points instead of 15). Russell, 3rd on the grid, dropped to 7th.

This raises the questions the rest of the notebook tries to answer: how did Leclerc gain
three positions and hold off the McLarens, and why did the fastest car not win?

## Laps

All the analyses in this notebook start from the laps data, so its structure is
inspected first.

In [3]:
race.laps

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:57:18.931000,LEC,16,0 days 00:01:28.179000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:29.989000,0 days 00:00:28.398000,NaT,0 days 00:56:50.708000,0 days 00:57:19.087000,316.0,313.0,311.0,300.0,False,MEDIUM,1.0,True,Ferrari,0 days 00:55:50.494000,NaT,1,2.0,False,,False,False
1,0 days 00:58:44.327000,LEC,16,0 days 00:01:25.396000,2.0,1.0,NaT,NaT,0 days 00:00:27.707000,0 days 00:00:29.265000,0 days 00:00:28.424000,0 days 00:57:46.661000,0 days 00:58:15.926000,0 days 00:58:44.350000,315.0,325.0,312.0,325.0,True,MEDIUM,2.0,True,Ferrari,0 days 00:57:18.931000,NaT,1,2.0,False,,False,True
2,0 days 01:00:09.506000,LEC,16,0 days 00:01:25.179000,3.0,1.0,NaT,NaT,0 days 00:00:27.679000,0 days 00:00:29.001000,0 days 00:00:28.499000,0 days 00:59:12.029000,0 days 00:59:41.030000,0 days 01:00:09.529000,313.0,324.0,313.0,328.0,True,MEDIUM,3.0,True,Ferrari,0 days 00:58:44.327000,NaT,1,2.0,False,,False,True
3,0 days 01:01:34.316000,LEC,16,0 days 00:01:24.810000,4.0,1.0,NaT,NaT,0 days 00:00:27.653000,0 days 00:00:28.883000,0 days 00:00:28.274000,0 days 01:00:37.182000,0 days 01:01:06.065000,0 days 01:01:34.339000,314.0,324.0,312.0,327.0,True,MEDIUM,4.0,True,Ferrari,0 days 01:00:09.506000,NaT,1,2.0,False,,False,True
4,0 days 01:02:58.919000,LEC,16,0 days 00:01:24.603000,5.0,1.0,NaT,NaT,0 days 00:00:27.630000,0 days 00:00:28.790000,0 days 00:00:28.183000,0 days 01:02:01.969000,0 days 01:02:30.759000,0 days 01:02:58.942000,313.0,325.0,312.0,321.0,True,MEDIUM,5.0,True,Ferrari,0 days 01:01:34.316000,NaT,1,2.0,False,,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,0 days 01:00:20.190000,TSU,22,0 days 00:01:26.665000,3.0,1.0,NaT,NaT,0 days 00:00:27.756000,0 days 00:00:29.819000,0 days 00:00:29.090000,0 days 00:59:21.321000,0 days 00:59:51.140000,0 days 01:00:20.230000,313.0,339.0,312.0,349.0,True,HARD,4.0,False,RB,0 days 00:58:53.525000,NaT,1,15.0,False,,False,True
1004,0 days 01:01:46.388000,TSU,22,0 days 00:01:26.198000,4.0,1.0,NaT,NaT,0 days 00:00:27.631000,0 days 00:00:29.689000,0 days 00:00:28.878000,0 days 01:00:47.861000,0 days 01:01:17.550000,0 days 01:01:46.428000,329.0,340.0,311.0,353.0,True,HARD,5.0,False,RB,0 days 01:00:20.190000,NaT,1,15.0,False,,False,True
1005,0 days 01:03:18.629000,TSU,22,0 days 00:01:32.241000,5.0,1.0,NaT,NaT,0 days 00:00:28.736000,0 days 00:00:32.722000,0 days 00:00:30.783000,0 days 01:02:15.164000,0 days 01:02:47.886000,0 days 01:03:18.669000,293.0,325.0,307.0,340.0,False,HARD,6.0,False,RB,0 days 01:01:46.388000,NaT,1,19.0,True,TRACK LIMITS AT TURN 2 LAP 5,False,True
1006,0 days 01:04:47.102000,TSU,22,0 days 00:01:28.473000,6.0,1.0,NaT,NaT,0 days 00:00:28.426000,0 days 00:00:30.441000,0 days 00:00:29.606000,0 days 01:03:47.095000,0 days 01:04:17.536000,0 days 01:04:47.142000,316.0,325.0,309.0,294.0,False,HARD,7.0,False,RB,0 days 01:03:18.629000,NaT,1,19.0,False,,False,True


Each row is one lap of one driver, ordered by driver (in finishing order) and lap number.
The 31 columns can be grouped by what they describe:

- **who and when**: `Driver`, `DriverNumber`, `Team`, `LapNumber`
- **lap and sector times**: `LapTime`, `Sector1Time`–`Sector3Time`
- **session timestamps**: `Time` (end of lap), `LapStartTime`, `Sector1SessionTime`–`Sector3SessionTime`, `LapStartDate`
- **speed traps**: `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST`
- **tyres**: `Stint`, `Compound`, `TyreLife`, `FreshTyre`
- **pit stops**: `PitInTime`, `PitOutTime`
- **race situation**: `Position`, `TrackStatus`
- **data quality**: `IsAccurate`, `Deleted`, `DeletedReason`, `FastF1Generated`, `IsPersonalBest`

Some special cases are already visible: the first lap has no sector 1 time and is marked
as inaccurate, some laps are deleted for track limits, and a lap ending in the pit lane
has a `PitInTime`.

To see how these columns tell the story of a race, the laps of the winner, Charles Leclerc,
are shown below with the most relevant columns.

In [4]:
race.laps.pick_drivers("LEC")[
    ["LapNumber", "LapTime", "Position", "Stint", "Compound", "TyreLife",
     "PitInTime", "PitOutTime", "TrackStatus"]
]

,LapNumber,LapTime,Position,Stint,Compound,TyreLife,PitInTime,PitOutTime,TrackStatus
0,1.0,0 days 00:01:28.179000,2.0,1.0,MEDIUM,1.0,NaT,NaT,1
1,2.0,0 days 00:01:25.396000,2.0,1.0,MEDIUM,2.0,NaT,NaT,1
2,3.0,0 days 00:01:25.179000,2.0,1.0,MEDIUM,3.0,NaT,NaT,1
3,4.0,0 days 00:01:24.810000,2.0,1.0,MEDIUM,4.0,NaT,NaT,1
4,5.0,0 days 00:01:24.603000,2.0,1.0,MEDIUM,5.0,NaT,NaT,1
5,6.0,0 days 00:01:24.663000,2.0,1.0,MEDIUM,6.0,NaT,NaT,1
6,7.0,0 days 00:01:24.434000,2.0,1.0,MEDIUM,7.0,NaT,NaT,1
7,8.0,0 days 00:01:24.569000,2.0,1.0,MEDIUM,8.0,NaT,NaT,1
8,9.0,0 days 00:01:24.362000,2.0,1.0,MEDIUM,9.0,NaT,NaT,1
9,10.0,0 days 00:01:24.432000,2.0,1.0,MEDIUM,10.0,NaT,NaT,1


Leclerc's laps already summarise his race:

- he gained two positions at the start, running 2nd after lap 1;
- his first stint on Medium tyres lasted 15 laps, with lap times rising from about 1:24.4
  to 1:25.6 in the final laps as the tyres degraded;
- the pit stop affects two laps: the in-lap (lap 15, with `PitInTime`) and the out-lap
  (lap 16, with `PitOutTime`), both much slower than a normal lap;
- he then ran 38 laps on Hard tyres with **no visible degradation**, between 1:23.2 and 1:23.9
  until the end;
- after the stop he dropped to 7th, then climbed back and took the lead on lap 39,
  likely as the cars ahead made their own stops (checked in the Positions section).

Of course, reading 53 rows is not practical: plotting lap times against lap number shows stints,
pit stops and tyre degradation at a glance. Lap times are converted to seconds,
and points are coloured by tyre compound following the F1 convention.

In [29]:
lec = race.laps.pick_drivers("LEC").copy()
lec["LapTimeSeconds"] = lec["LapTime"].dt.total_seconds()

fig = px.line(
    lec,
    x="LapNumber",
    y="LapTimeSeconds",
    color="Compound",
    markers=True,
    color_discrete_map=compound_colors,
    hover_data=["Position", "TyreLife"],
    labels={"LapNumber": "Lap", "LapTimeSeconds": "Lap time (s)", "TyreLife": "Tyre age"},
    title="Leclerc – lap times, 2024 Italian GP",
    template="plotly_dark",
)
fig.add_vline(x=15.5, line_dash="dash", line_color="gray", annotation_text="Pit stop", annotation_position="top")
fig.show()

The plot shows the three phases of Leclerc's race: the Medium stint, with lap times rising
in the last laps as the tyres degraded; the pit stop, with the slow in-lap and out-lap;
and the long Hard stint, remarkably consistent until the end.

The pit stop laps, however, compress the y-axis and hide the smaller variations
that matter for pace and degradation. They need to be removed before comparing drivers.

### Representative laps

As mentioned, not every lap reflects a driver's pace. Before comparing drivers, laps affected by the start,
pit stops, flags or data issues are identified and counted.

In [13]:
laps = race.laps

issues = pd.DataFrame({
    "First lap": laps["LapNumber"] == 1,
    "Pit in/out lap": laps["PitInTime"].notna() | laps["PitOutTime"].notna(),
    "Not green flag": laps["TrackStatus"] != "1",
    "Inaccurate": laps["IsAccurate"] == False,
    "Deleted": laps["Deleted"] == True,
})

print("Laps with at least one issue:", issues.any(axis=1).sum(), "of", len(laps))
issues.sum()

Laps with at least one issue: 94 of 1008


First lap         20
Pit in/out lap    61
Not green flag     0
Inaccurate        81
Deleted           13
dtype: int64

About 9% of the laps have at least one issue. No lap was run under yellow flag or safety car,
and the categories overlap heavily (94 laps in total against a sum of 175).
Are the inaccurate laps simply the first lap and the pit laps, or something else?

In [14]:
other_inaccurate = issues["Inaccurate"] & ~issues["First lap"] & ~issues["Pit in/out lap"]
laps[other_inaccurate][["Driver", "LapNumber", "LapTime", "PitInTime", "PitOutTime"]]

,Driver,LapNumber,LapTime,PitInTime,PitOutTime


All inaccurate laps are either first laps or pit laps: FastF1 flags exactly the laps
that do not represent normal racing. The laps kept for pace analysis are therefore those
with none of the issues above: no first lap, no pit in/out lap, green flag only,
accurate timing and not deleted.

In [15]:
clean = laps[~issues.any(axis=1)].copy()
clean["LapTimeSeconds"] = clean["LapTime"].dt.total_seconds()

print(len(clean), "clean laps")
clean["LapTimeSeconds"].describe()

914 clean laps


count    914.000000
mean      84.784802
std        1.247637
min       81.432000
25%       83.756500
50%       84.844000
75%       85.660500
max       89.256000
Name: LapTimeSeconds, dtype: float64

Most clean laps fall within a two-second window around the median (84.8 s), but the slowest
lap is more than four seconds slower. The slowest laps are inspected below to see whether
they are genuine outliers or simply slower cars.

In [16]:
clean.nlargest(10, "LapTimeSeconds")[
    ["Driver", "Team", "LapNumber", "LapTimeSeconds", "Position", "Compound", "TyreLife"]
]

,Driver,Team,LapNumber,LapTimeSeconds,Position,Compound,TyreLife
901,ZHO,Kick Sauber,5.0,89.256,20.0,MEDIUM,6.0
898,ZHO,Kick Sauber,2.0,88.721,20.0,MEDIUM,3.0
1006,TSU,RB,6.0,88.473,19.0,HARD,7.0
794,BOT,Kick Sauber,2.0,88.377,19.0,HARD,3.0
950,STR,Aston Martin,2.0,88.342,18.0,HARD,3.0
797,BOT,Kick Sauber,5.0,88.242,17.0,HARD,6.0
824,BOT,Kick Sauber,32.0,87.859,15.0,HARD,33.0
584,COL,Williams,2.0,87.845,17.0,MEDIUM,2.0
881,HUL,Haas F1 Team,37.0,87.809,19.0,HARD,5.0
938,ZHO,Kick Sauber,42.0,87.640,17.0,HARD,28.0


The slowest clean laps belong to cars at the back of the field, mostly in the opening laps
with full fuel and heavy traffic. They are genuine laps rather than data errors. A global
threshold such as 107% of the fastest lap would mainly remove laps of the slowest cars,
biasing the comparison against them.

No further filter is applied: pace is compared using the **median** lap time of each driver,
which is robust to a few isolated slow laps.

These rules are implemented in the `clean_laps` function of the project's `cleaning` module,
so that the same selection is used throughout the notebook and the app. Applying it to the
race should return the same 914 laps obtained above.

In [18]:
clean = clean_laps(race.laps)
len(clean)

914

## Race pace

Comparing raw lap times is unfair when drivers ran at different moments of the race:
early laps are slower for everyone because of fuel load and traffic. Each clean lap is
therefore compared with the median lap time of the whole field **on the same lap**.
A driver's race pace is the median of these differences: negative values mean faster
than the field.

In [23]:
MIN_CLEAN_LAPS = 10

clean["FieldLapMedian"] = clean.groupby("LapNumber")["LapTimeSeconds"].transform("median")
clean["DeltaToField"] = clean["LapTimeSeconds"] - clean["FieldLapMedian"]

pace = (
    clean.groupby(["Driver", "Team"])
    .agg(PaceDelta=("DeltaToField", "median"), CleanLaps=("DeltaToField", "count"))
    .reset_index()
    .sort_values("PaceDelta")
)
pace["Label"] = pace["Driver"].where(pace["CleanLaps"] >= MIN_CLEAN_LAPS, pace["Driver"] + "*")
pace

,Driver,Team,PaceDelta,CleanLaps,Label
12,PIA,McLaren,-1.27775,48,PIA
9,NOR,McLaren,-1.25100,47,NOR
7,LEC,Ferrari,-0.96775,50,LEC
5,HAM,Mercedes,-0.90050,48,HAM
14,RUS,Mercedes,-0.77400,47,RUS
18,VER,Red Bull Racing,-0.73875,48,VER
15,SAI,Ferrari,-0.59350,49,SAI
11,PER,Red Bull Racing,-0.51525,48,PER
1,ALO,Aston Martin,0.00000,47,ALO
6,HUL,Haas F1 Team,0.10600,47,HUL


Relative to the field, the McLarens had the strongest race pace, about 1.25 s per lap
faster than the field median. Leclerc ranks 3rd, ahead of Hamilton and Verstappen,
but still about 0.3 s per lap slower than the McLarens: over the race, raw pace alone
does not explain his win, which points to strategy.

Tsunoda's value is based on only four clean laps, all in the opening phase of the race
before his retirement, so it is shown but marked as less reliable.

This measure does not yet account for tyres: drivers who stopped twice ran on fresher
tyres on average, which flatters their pace. This is examined in the tyre strategy section.

At this point, the gaps are easier to compare in a chart. Bars are coloured with each team's official
colour, taken from the session results.

In [24]:
team_colors = {
    team: f"#{color}"
    for team, color in race.results[["TeamName", "TeamColor"]].drop_duplicates().values
}

fig = px.bar(
    pace,
    x="PaceDelta",
    y="Label",
    color="Team",
    orientation="h",
    color_discrete_map=team_colors,
    category_orders={"Label": pace["Label"].tolist()},
    hover_data={"Label": False, "Driver": True, "PaceDelta": ":.3f", "CleanLaps": True},
    labels={"PaceDelta": "Median gap to the field on the same lap (s)", "Label": "",
            "CleanLaps": "Clean laps"},
    title="Race pace relative to the field – 2024 Italian Grand Prix",
    template="plotly_dark",
    height=600,
)
fig.add_vline(x=0, line_color="gray")
fig.add_annotation(text=f"* fewer than {MIN_CLEAN_LAPS} clean laps", showarrow=False,
                   xref="paper", yref="paper", x=1, y=-0.12)
fig.show()

## Tyre strategy

A strategy is a sequence of stints, each run on one tyre compound. For every driver,
stints are summarised by compound, first and last lap, and number of laps.
Unlike the pace analysis, all laps are used here rather than the clean ones only:
reconstructing a strategy requires the first lap and the pit laps as well.
Drivers are listed in finishing order.

In [25]:
driver_order = race.results["Abbreviation"].tolist()

stints = (
    race.laps.groupby(["Driver", "Stint", "Compound"])
    .agg(StartLap=("LapNumber", "min"), EndLap=("LapNumber", "max"), Laps=("LapNumber", "count"))
    .reset_index()
)
stints["Driver"] = pd.Categorical(stints["Driver"], categories=driver_order, ordered=True)
stints = stints.sort_values(["Driver", "Stint"])
stints

,Driver,Stint,Compound,StartLap,EndLap,Laps
18,LEC,1.0,MEDIUM,1.0,15.0,15
19,LEC,2.0,HARD,16.0,53.0,38
30,PIA,1.0,MEDIUM,1.0,16.0,16
31,PIA,2.0,HARD,17.0,38.0,22
32,PIA,3.0,HARD,39.0,53.0,15
22,NOR,1.0,MEDIUM,1.0,14.0,14
23,NOR,2.0,HARD,15.0,32.0,18
24,NOR,3.0,HARD,33.0,53.0,21
38,SAI,1.0,MEDIUM,1.0,19.0,19
39,SAI,2.0,HARD,20.0,53.0,34


The table confirms the strategic split at the front: both Ferraris made a single stop
(Medium to Hard), with Leclerc running 38 laps on the same set of Hard tyres, while both
McLarens stopped twice (Medium, Hard, Hard). Among the top eight, only the Ferraris
one-stopped. Further back, Stroll made three stops, fitting Soft tyres for the last two laps.
Lapped drivers' final stints end on lap 52.

The same information is clearer as a strategy chart: one row per driver, in finishing
order, with each stint drawn as a bar spanning its laps and coloured by compound.

In [30]:
stints["StartFrom"] = stints["StartLap"] - 1

fig = px.bar(
    stints,
    x="Laps",
    y="Driver",
    base="StartFrom",
    color="Compound",
    orientation="h",
    color_discrete_map=compound_colors,
    category_orders={"Driver": driver_order},
    hover_data={"Stint": True, "StartLap": True, "EndLap": True, "Laps": True, "StartFrom": False},
    labels={"Laps": "Lap", "Driver": ""},
    title="Tyre strategies – 2024 Italian Grand Prix",
    template="plotly_dark",
    height=600,
)
fig.update_layout(barmode="overlay")
fig.update_traces(marker_line_color="#111111", marker_line_width=2)
fig.show()

### Strategies in practice

How did a one-stop strategy beat a two-stop one? The fight for the win was between
Leclerc and Piastri: plotting their clean lap times shows how their pace evolved
with tyre age across their stints. Pit stops are shown as shaded areas covering the
in-lap and the out-lap, which are excluded here since they include time spent in the
pit lane. These laps are where undercuts are decided, so they are analysed separately
in the pit stops section.

In [36]:
duel = clean.pick_drivers(["LEC", "PIA"])
driver_colors = {"LEC": team_colors["Ferrari"], "PIA": team_colors["McLaren"]}

fig = px.line(
    duel,
    x="LapNumber",
    y="LapTimeSeconds",
    color="Driver",
    markers=True,
    color_discrete_map=driver_colors,
    hover_data=["Compound", "TyreLife"],
    labels={"LapNumber": "Lap", "LapTimeSeconds": "Lap time (s)", "TyreLife": "Tyre age"},
    title="Leclerc vs Piastri – clean lap times, 2024 Italian Grand Prix",
    template="plotly_dark",
)

duel_stints = stints[stints["Driver"].isin(["LEC", "PIA"])]
pit_stops = duel_stints[
    duel_stints["Stint"] < duel_stints.groupby("Driver", observed=True)["Stint"].transform("max")
]
for _, stop in pit_stops.iterrows():
    fig.add_vrect(
        x0=stop["EndLap"] - 0.5,
        x1=stop["EndLap"] + 1.5,
        fillcolor=driver_colors[stop["Driver"]],
        opacity=0.15,
        line_width=0,
    )
    fig.add_annotation(
        x=stop["EndLap"] + 0.5,
        y=1.02 if stop["Driver"] == "LEC" else 1.08,
        yref="paper",
        text=f"{stop['Driver']} pit",
        showarrow=False,
    )
fig.show()

In [38]:
t = race.laps.pick_drivers(["LEC", "PIA"]).copy()
t["LapTimeSeconds"] = t["LapTime"].dt.total_seconds()
t["TimeSeconds"] = t["Time"].dt.total_seconds()

table = t.pivot(index="LapNumber", columns="Driver",
                values=["LapTimeSeconds", "Compound", "TyreLife", "Position", "TimeSeconds"])
table[("LapDiff", "PIA-LEC")] = table[("LapTimeSeconds", "PIA")] - table[("LapTimeSeconds", "LEC")]
table[("Gap", "PIA-LEC")] = table[("TimeSeconds", "PIA")] - table[("TimeSeconds", "LEC")]
table = table.drop(columns="TimeSeconds")

print(table.loc[35:].round(3).to_string())

          LapTimeSeconds          Compound       TyreLife       Position      LapDiff     Gap
Driver               LEC      PIA      LEC   PIA      LEC   PIA      LEC  PIA PIA-LEC PIA-LEC
LapNumber                                                                                    
35.0              83.681   83.409     HARD  HARD     20.0  19.0      2.0  1.0  -0.272  -5.856
36.0              83.714   83.746     HARD  HARD     21.0  20.0      2.0  1.0   0.032  -5.824
37.0               83.55   83.691     HARD  HARD     22.0  21.0      2.0  1.0   0.141  -5.683
38.0              83.624   88.125     HARD  HARD     23.0  22.0      2.0  1.0   4.501  -1.182
39.0              83.718  103.438     HARD  HARD     24.0   1.0      1.0  3.0   19.72  18.538
40.0               83.69   82.264     HARD  HARD     25.0   2.0      1.0  3.0  -1.426  17.112
41.0              83.611   82.066     HARD  HARD     26.0   3.0      1.0  3.0  -1.545  15.567
42.0               83.67   82.331     HARD  HARD     27.0   

The two drivers had almost identical pace in the opening laps. From lap 10 Leclerc's
Medium tyres degraded faster, and Ferrari pitted him first, on lap 15, one lap before Piastri.
On the Hard tyres Piastri was slightly quicker in the middle of the race, until his tyres
aged and the two settled at similar lap times.

The race was decided in the final phase. Piastri led by almost 6 seconds when he made
his second stop on lap 38, which cost him about 24 seconds relative to Leclerc: he rejoined
18.5 seconds behind, in third place. On fresh tyres he was about 1.1 seconds per lap faster
on average, while Leclerc, on Hard tyres more than 20 laps old, kept lapping consistently
between 83.5 and 83.7 seconds. Piastri recovered almost 16 seconds in 14 laps, including
a slower lap to overtake Norris, and finished 2.7 seconds behind.

### Tyre degradation

Degradation is estimated for each stint as the slope of a straight line fitted to lap time
against tyre age: the seconds gained or lost per additional lap on the same tyres.
Since the car also gets lighter as fuel burns, the slope measures **net** degradation
(tyre wear minus the fuel effect), and can be negative. Only stints with at least
8 clean laps are considered.

In [41]:
MIN_STINT_LAPS = 8

def degradation(stint):
    if len(stint) < MIN_STINT_LAPS:
        return pd.Series({"Laps": len(stint), "SlopePerLap": np.nan})
    slope = np.polyfit(stint["TyreLife"], stint["LapTimeSeconds"], 1)[0]
    return pd.Series({"Laps": len(stint), "SlopePerLap": slope})

deg = (
    clean.groupby(["Driver", "Team", "Stint", "Compound"])
    .apply(degradation, include_groups=False)
    .reset_index()
)
deg = deg.dropna(subset=["SlopePerLap"])
deg[deg["Driver"].isin(["LEC", "PIA", "NOR"])]

,Driver,Team,Stint,Compound,Laps,SlopePerLap
18,LEC,Ferrari,1.0,MEDIUM,13.0,0.019940
19,LEC,Ferrari,2.0,HARD,37.0,-0.008320
22,NOR,McLaren,1.0,MEDIUM,12.0,-0.041287
23,NOR,McLaren,2.0,HARD,15.0,-0.098032
24,NOR,McLaren,3.0,HARD,20.0,-0.056423
30,PIA,McLaren,1.0,MEDIUM,14.0,-0.009451
31,PIA,McLaren,2.0,HARD,20.0,-0.033548
32,PIA,McLaren,3.0,HARD,14.0,-0.005989


Almost all slopes are close to zero or negative: at Monza, the fuel effect offset most of
the tyre wear, so lap times did not get slower as tyres aged. The only clearly positive slope
is Leclerc's Medium stint (+0.02 s per lap), consistent with the drop in pace seen before
his stop.

Leclerc's Hard stint was essentially flat over 37 laps (about −0.01 s per lap). This is
what made the one-stop strategy possible: his tyres never fell away. The McLarens' Hard
stints show more negative slopes, suggesting slightly lower wear, although slopes on
stints of 15–20 laps are sensitive to a few slower laps in traffic. Since the fuel effect
is similar for all cars, a flat net slope implies that Leclerc's tyre wear roughly matched
the fuel gain: accumulated over 25 laps, this is consistent with the one-second advantage
Piastri enjoyed on fresh tyres at the end.

## Pit stops

A pit stop costs time on two laps: the in-lap, which ends in the pit lane, and the out-lap,
which starts from it. The time lost is estimated by comparing these two laps with the driver's
typical lap time around the stop, given by the median of their clean laps within five laps
before and after.

In [42]:
all_laps = race.laps.copy()
all_laps["LapTimeSeconds"] = all_laps["LapTime"].dt.total_seconds()

in_laps = all_laps[all_laps["PitInTime"].notna()][["Driver", "Team", "LapNumber", "LapTimeSeconds"]]
out_laps = all_laps[all_laps["PitOutTime"].notna()][["Driver", "LapNumber", "LapTimeSeconds"]]
out_laps["LapNumber"] -= 1  # align each out-lap with the in-lap before it

stops = in_laps.merge(out_laps, on=["Driver", "LapNumber"], suffixes=("In", "Out"))

def reference_lap(stop):
    nearby = clean[
        (clean["Driver"] == stop["Driver"])
        & clean["LapNumber"].between(stop["LapNumber"] - 5, stop["LapNumber"] + 6)
    ]
    return nearby["LapTimeSeconds"].median()

stops["ReferenceLap"] = stops.apply(reference_lap, axis=1)
stops["PitLoss"] = stops["LapTimeSecondsIn"] + stops["LapTimeSecondsOut"] - 2 * stops["ReferenceLap"]
stops.sort_values("PitLoss")

,Driver,Team,LapNumber,LapTimeSecondsIn,LapTimeSecondsOut,ReferenceLap,PitLoss
28,STR,Aston Martin,36.0,89.334,103.775,84.7490,23.611
16,ALO,Aston Martin,12.0,89.740,105.594,85.7580,23.818
3,NOR,McLaren,14.0,89.417,103.840,84.4195,24.418
20,OCO,Alpine,31.0,90.757,104.710,85.5160,24.435
7,HAM,Mercedes,37.0,88.349,102.426,83.1700,24.435
23,BOT,Kick Sauber,33.0,90.924,105.462,85.9040,24.578
17,ALO,Aston Martin,35.0,89.799,104.152,84.4430,25.065
4,NOR,McLaren,32.0,87.414,104.073,83.1370,25.213
13,PER,Red Bull Racing,35.0,88.255,104.074,83.5425,25.244
12,PER,Red Bull Racing,23.0,89.869,105.017,84.7785,25.329


Most stops cost between 23.5 and 26.5 seconds, the typical pit loss at Monza, including those
of the leading drivers (Leclerc 26.4 s, Piastri 25.3 and 25.8 s). A few stops cost
considerably more; lap times alone do not tell whether this was due to a slow stop,
a repair or traffic on the out-lap.

This allows a rough strategic balance: an extra stop cost the McLarens about 25 seconds,
while Piastri's pace advantage over Leclerc, about 0.3 seconds per lap, was worth about
15 seconds over the race. Faster pace was not enough to pay for the extra stop.

The effect of the two strategies is clearest in the gap between Leclerc and Piastri at the end
of each lap. Positive values mean Piastri is behind Leclerc; shaded areas mark the pit stops.

In [43]:
lap_end = all_laps.pivot(index="LapNumber", columns="Driver", values="Time")
gap = (lap_end["PIA"] - lap_end["LEC"]).dt.total_seconds().rename("Gap").reset_index()

fig = px.line(
    gap,
    x="LapNumber",
    y="Gap",
    markers=True,
    labels={"LapNumber": "Lap", "Gap": "Piastri's gap to Leclerc (s)"},
    title="Gap between Piastri and Leclerc – 2024 Italian Grand Prix",
    template="plotly_dark",
)
fig.update_traces(line_color="#f0f0ec")
fig.add_hline(y=0, line_color="gray")

for _, stop in pit_stops.iterrows():
    fig.add_vrect(
        x0=stop["EndLap"] - 0.5,
        x1=stop["EndLap"] + 1.5,
        fillcolor=driver_colors[stop["Driver"]],
        opacity=0.15,
        line_width=0,
    )
    fig.add_annotation(
        x=stop["EndLap"] + 0.5,
        y=1.02 if stop["Driver"] == "LEC" else 1.08,
        yref="paper",
        text=f"{stop['Driver']} pit",
        showarrow=False,
    )
fig.update_layout(margin_t=110)
fig.show()

The gap summarises the whole race. Piastri led from the start and gradually pulled away
to about 3.4 seconds as Leclerc's Medium tyres faded. The spike around laps 15–16 is an
effect of the staggered stops: for one lap Leclerc had already paid for his stop and
Piastri had not. Once both had stopped, the gap was again about 3.4 seconds, so Leclerc's
earlier stop gained him no time, but it relieved him of degrading tyres.

Piastri then extended his lead to almost 6 seconds before his second stop on lap 38,
which put him 18.5 seconds behind. From there he recovered more than one second per lap,
but crossed the line 2.7 seconds behind: the race was decided by the 25 seconds spent
in the pit lane for the extra stop.

## Positions

Positions at the end of each lap show how the race order evolved, starting from the grid
(lap 0). The three podium finishers are highlighted, while the rest of the field is shown
in grey for context.

In [45]:
grid = (
    race.results[["Abbreviation", "GridPosition"]]
    .rename(columns={"Abbreviation": "Driver", "GridPosition": "Position"})
    .assign(LapNumber=0)
)
positions = pd.concat([grid, all_laps[["Driver", "LapNumber", "Position"]]], ignore_index=True)

highlight = {"LEC": team_colors["Ferrari"], "PIA": team_colors["McLaren"], "NOR": team_colors["McLaren"]}
colors = {driver: highlight.get(driver, "#555555") for driver in driver_order}

fig = px.line(
    positions,
    x="LapNumber",
    y="Position",
    color="Driver",
    color_discrete_map=colors,
    category_orders={"Driver": driver_order},
    labels={"LapNumber": "Lap"},
    title="Positions lap by lap – 2024 Italian Grand Prix",
    template="plotly_dark",
    height=600,
)
fig.update_yaxes(range=[20.5, 0.5], dtick=1)
fig.for_each_trace(
    lambda trace: trace.update(
        line_width=3 if trace.name in highlight else 1,
        line_dash="dash" if trace.name == "NOR" else "solid",
    )
)
fig.show()

At the start, both Piastri and Leclerc passed Norris, who dropped from pole position to 3rd.
During the first round of stops (laps 15–22) the order was temporarily reshuffled by drivers
who had started on Hard tyres and had not yet stopped. Once it settled, Piastri led Norris
and Leclerc. Leclerc moved to 2nd when Norris made his second stop on lap 32, and took the
lead when Piastri stopped on lap 38; Piastri rejoined 3rd and passed Norris on lap 45.

After the opening lap, Leclerc never overtook either McLaren on track: every position he
gained came from their pit stops. The win was decided by strategy, not by on-track overtaking.